# 🪰 RAPID v3 — Block-wise WTA + Prototype Classifier trên CIFAR-100
Notebook này clone repo, chạy `main.py` với đúng hyperparameter,
và hiển thị Accuracy Matrix + Accumulated Accuracy.

**Cải tiến v3:** Block-wise Absolute WTA · NCM Prototype Classifier · [-1,1] normalization

In [ ]:
# ── Cell 1: Cài thư viện ──────────────────────────────────────────────────────
!pip install timm==0.9.16 -q

In [ ]:
# ── Cell 2: Clone repo & setup ────────────────────────────────────────────────
import os

REPO_URL = "https://github.com/ZaPhat206/LAB_FLY.git"
WORK_DIR = "/kaggle/working/LAB_FLY"

if os.path.isdir(WORK_DIR):
    print("Repo already cloned, pulling latest...")
    !git -C {WORK_DIR} pull -q
else:
    !git clone {REPO_URL} {WORK_DIR} -q

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())
print("Files:", os.listdir("."))

In [ ]:
# ── Cell 3: Tải pretrained ViT-B/16 ──────────────────────────────────────────
import os
os.chdir(f"{WORK_DIR}/pretrained_model")
!sh download.sh
os.chdir(WORK_DIR)

In [ ]:
# ── Cell 4: Fix xung đột tên package với Kaggle ───────────────────────────────
# Kaggle có sẵn package 'datasets' (HuggingFace) → rename folder để tránh conflict
import os, shutil
os.chdir(WORK_DIR)

if os.path.isdir("datasets") and not os.path.isdir("my_datasets"):
    shutil.move("datasets", "my_datasets")
    print("Renamed: datasets → my_datasets")
else:
    print("Folder rename: already done or not needed.")

# Sửa import trong main.py
!sed -i 's/from datasets.load_dataset/from my_datasets.load_dataset/g' main.py
# Sửa import trong my_datasets/__init__.py nếu có
!find my_datasets -name '*.py' -exec sed -i 's/from datasets/from my_datasets/g' {} + 2>/dev/null || true
print("Import fixed.")

In [ ]:
# ── Cell 5: Chạy RAPID v3 trên CIFAR-100 ─────────────────────────────────────
# Hyperparameter theo paper FLY-CL (Appendix C.3)
# LƯU Ý: main.py v3 đã xóa --ridge_lower / --ridge_upper → không truyền vào
import subprocess, sys, os
os.chdir(WORK_DIR)

CMD = [
    sys.executable, "main.py",
    "--dataset",           "CIFAR-100",
    "--root",              "../data",
    "--num_classes",       "100",
    "--num_tasks",         "10",
    "--model_name",        "vit_base_patch16_224",
    "--embedding_dim",     "768",
    "--expand_dim",        "10000",
    "--synaptic_degree",   "300",
    "--coding_level",      "0.3",
    "--fisher_block",      "512",
    "--fisher_sat",        "0.005",
    "--data_augmentation", "vit",   # ← [-1,1] normalization (Paper Appendix C.3)
    "--seed",              "1993",
    "--batch_size",        "128",
    "--gpu",               "0",
]

print("▶ Running RAPID v3 on CIFAR-100 ...\n")
result = subprocess.run(CMD, capture_output=True, text=True, cwd=WORK_DIR)

# Lưu output để parse
OUT_FILE = "/kaggle/working/rapid_cifar_output.txt"
with open(OUT_FILE, "w") as f:
    f.write(result.stdout)
    if result.stderr:
        f.write("\n--- STDERR ---\n" + result.stderr)

if result.returncode != 0:
    print("❌ ERROR (last 3000 chars):")
    print(result.stderr[-3000:])
else:
    print(result.stdout)

In [ ]:
# ── Cell 6: Parse kết quả & hiển thị đẹp ─────────────────────────────────────
import re
import numpy as np

with open("/kaggle/working/rapid_cifar_output.txt") as f:
    txt = f.read()

# --- Parse Accuracy Matrix ---
# Format mới: mỗi row là list Python [x, y, z, ...]
acc_matrix = []
in_block = False
for line in txt.splitlines():
    if "Accuracy Matrix" in line:
        in_block = True
        continue
    if in_block:
        line = line.strip()
        if line.startswith("["):
            try:
                acc_matrix.append(eval(line))
            except:
                pass
        elif line == "" and acc_matrix:
            in_block = False

# --- Parse scalar metrics (format mới: inline trên cùng dòng) ---
def parse_scalar(pattern):
    m = re.search(pattern, txt, re.DOTALL)
    return float(m.group(1)) if m else None

# Format mới: "Accumulated Accuracy (A_bar): XX.XX"
accumulated_acc = parse_scalar(r"Accumulated Accuracy.*?:\s*([\d.]+)")
avg_train_time  = parse_scalar(r"Avg Training Time:\s*([\d.]+)")
avg_feat_time   = parse_scalar(r"Avg Extract Time:\s*([\d.]+)")

# --- Tính Forgetting (BWT) ---
forgetting_list = []
if acc_matrix:
    T = len(acc_matrix)
    for i in range(T - 1):
        row   = acc_matrix[i]
        peak  = float(row[i])   if isinstance(row[i],   (int, float)) else 0.0
        final = float(row[T-1]) if isinstance(row[T-1], (int, float)) and row[T-1] != "0.00" else 0.0
        if peak > 0:
            forgetting_list.append(peak - final)

mean_forgetting = np.mean(forgetting_list) if forgetting_list else 0.0

# --- Hiển thị bảng kết quả ---
print("═" * 52)
print(f"{'🪰 RAPID v3  —  CIFAR-100  (10 tasks)':^52}")
print("═" * 52)
print(f"  {'Accumulated Accuracy':<30} {accumulated_acc:>8.2f} %")
print(f"  {'Forgetting (BWT)':<30} {mean_forgetting:>8.2f} %")
print(f"  {'Avg Training Time / task':<30} {avg_train_time:>8.2f} s")
print(f"  {'Avg Feature Extract Time':<30} {avg_feat_time:>8.2f} s")
print("─" * 52)

# --- In Accuracy Matrix dạng bảng ---
if acc_matrix:
    T = len(acc_matrix)
    header = "Task  │ " + " │ ".join(f"T{j+1:02d}" for j in range(T))
    print("\n" + header)
    print("─" * len(header))
    for i, row in enumerate(acc_matrix):
        cells = []
        for j, v in enumerate(row):
            if isinstance(v, (int, float)) and v > 0:
                cells.append(f"{v:5.2f}")
            else:
                cells.append("  —  ")
        print(f"  T{i+1:02d} │ " + " │ ".join(cells))
print("═" * 52)

In [ ]:
# ── Cell 7 (Optional): Vẽ biểu đồ ───────────────────────────────────────────
import matplotlib.pyplot as plt

# Parse A_t từ format mới: "  Task 0: XX.XX"
avg_accs = []
for line in txt.splitlines():
    m = re.match(r"\s*Task\s+\d+:\s*([\d.]+)", line)
    if m:
        avg_accs.append(float(m.group(1)))

if avg_accs and accumulated_acc is not None:
    tasks = list(range(1, len(avg_accs) + 1))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("RAPID v3 — CIFAR-100 (10 tasks)", fontsize=14, fontweight="bold")

    # Plot 1: Average Accuracy theo task
    axes[0].plot(tasks, avg_accs, "r-o", linewidth=2, markersize=7, label="RAPID v3")
    axes[0].axhline(y=accumulated_acc, linestyle="--", color="gray",
                    label=f"A_bar = {accumulated_acc:.2f}%")
    axes[0].set_xlabel("Task")
    axes[0].set_ylabel("Average Accuracy (%)")
    axes[0].set_title("Incremental Avg Accuracy (A_t)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xticks(tasks)

    # Plot 2: Forgetting per task
    if forgetting_list:
        task_ids = list(range(1, len(forgetting_list) + 1))
        axes[1].bar(task_ids, forgetting_list, color="tomato", alpha=0.85, edgecolor="darkred")
        axes[1].axhline(y=mean_forgetting, linestyle="--", color="black",
                        label=f"Mean = {mean_forgetting:.2f}%")
        axes[1].set_xlabel("Task")
        axes[1].set_ylabel("Forgetting (%)")
        axes[1].set_title("Per-Task Forgetting")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3, axis="y")
        axes[1].set_xticks(task_ids)

    plt.tight_layout()
    plt.savefig("/kaggle/working/rapid_cifar_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Plot saved → /kaggle/working/rapid_cifar_results.png")
else:
    print("Không tìm thấy dữ liệu để vẽ. Kiểm tra output file.")